In [1]:
from IPython.display import clear_output
from Buy_Presure_Scaner import *
from Volatile_4h_Scaner import *
from Gainer_List import *
from Consolidation_24_Scaner import *
from RSI_30_Scaner import *
from Fetch_Live_Data import *
from Chandelier_ZLSMA import *
from Trade_Execution import *
#from Trade_Execution_with_Testnet import *

In [10]:
def get_max_buy_pressure_trend_symbols():
    """
    Filter cryptocurrency data and return symbols with maximum buy pressure trend.
    
    Parameters:
    binance_df_1_hr (DataFrame): Input DataFrame with cryptocurrency data
    
    Returns:
    numpy.ndarray: Array of symbols with maximum buy pressure trend
    """
    binance_df_1_hr = get_binance_buy_presure(min_volume_usdt=1000000, top_n=50)
    # Define stablecoins to filter out
    stablecoins = [ 
        'GUSDUSDT', 'FRAXUSDT', 'USDDUSDT', 'MIMUSDT', 'LUSDUSDT', 'FEIUSDT', 'HUSDUSDT', 'SUSDUSDT', 'OUSDUSDT', 
        'USTCUSDT', 'VAIUSDT', 'DOLAUSDT', 'ALUSDUSDT', 'MUSDUSDT', 'DUSDUSDT', 'CUSDUSDT', 'NUSDUSDT', 'ZUSDUSDT', 'EURUSDT'
    ]
    
    # Apply filters
    binance_df_1_hr_filtered = binance_df_1_hr[ 
        (binance_df_1_hr['current_buy_pressure'] > binance_df_1_hr['current_sell_pressure']) &  
        (binance_df_1_hr['momentum'] != "BUILDING") &  
        (binance_df_1_hr['buy_pressure_trend'] > 1) &  
        (binance_df_1_hr['volume_trend'] > 1) 
    ]
    
    # Select relevant columns
    binance_df_1_hr_filtered = binance_df_1_hr_filtered[["symbol", "current_buy_pressure", "avg_buy_pressure_5periods", "buy_pressure_trend", "signal", "momentum"]]
    
    # Filter out stablecoins
    binance_df_1_hr_filtered = binance_df_1_hr_filtered[~binance_df_1_hr_filtered['symbol'].isin(stablecoins)]

    # Check if any data remains after filtering
    if binance_df_1_hr_filtered.empty:
        return []
    
    print(binance_df_1_hr_filtered)
    print("\n")
    
    # Find maximum buy pressure trend
    max_buy_pressure_trend = binance_df_1_hr_filtered['buy_pressure_trend'].max()
    max_buy_pressure_trend_row = binance_df_1_hr_filtered[binance_df_1_hr_filtered['buy_pressure_trend'] == max_buy_pressure_trend]
    # Return symbol values
    return max_buy_pressure_trend_row['symbol'].values

In [11]:
symbols = get_max_buy_pressure_trend_symbols()

       symbol  current_buy_pressure  avg_buy_pressure_5periods  \
1    KMNOUSDT                 80.03                      58.91   
3     TIAUSDT                 81.56                      58.03   
14   PNUTUSDT                 79.53                      54.50   
16   LINKUSDT                 60.91                      53.92   
20   ATOMUSDT                 71.24                      52.90   
21    JUPUSDT                 72.93                      52.47   
23   SHIBUSDT                 68.89                      52.38   
25   DOGEUSDT                 73.75                      52.32   
27  TRUMPUSDT                 72.48                      52.22   
31   GALAUSDT                 73.18                      51.99   
32    VETUSDT                 78.82                      51.97   
33  PENGUUSDT                 57.44                      51.90   
45    TONUSDT                 61.46                      49.96   

    buy_pressure_trend signal   momentum  
1                 5.67    BUY  E

In [12]:
symbols

array(['TRUMPUSDT'], dtype=object)

In [14]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 100)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[90:]

In [ ]:
def run_task():
    # Get the latest symbols every 2 hours
    symbols = get_max_buy_pressure_trend_symbols()
    return symbols

def fetch_and_display_data(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours
    print("New symbols received. Monitoring begins...\n")
    
    # Continuous 10-second updates with fetched data
    while True:
        out = fetch_and_display_data(symbols)

        # Check if the DataFrame is not empty and get the last row
        if not out.empty:
            # Access the last row using iloc[-1]
            if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                print("Buy signal detected for symbols:", symbols)
            else:
                print("No buy signal detected for symbols:", symbols)
        else:
            print("The DataFrame is empty. No data available.")
        
        time.sleep(10)  # Wait for 10 seconds before running the next iteration
    
    # Sleep for 2 hours before getting new symbols
    time.sleep(2 * 60 * 60)  # Sleep for 2 hours (in seconds)

Monitoring on:  ['ONDOUSDT']
             timestamp   close  zlsma_200  buy_signal  sell_signal
90 2025-06-12 19:15:00  0.8393   0.840546           0            1
91 2025-06-12 19:30:00  0.8388   0.839852           0            1
92 2025-06-12 19:45:00  0.8405   0.839731           0            1
93 2025-06-12 20:00:00  0.8386   0.839389           0            1
94 2025-06-12 20:15:00  0.8331   0.837737           0            1
95 2025-06-12 20:30:00  0.8366   0.836853           0            1
96 2025-06-12 20:45:00  0.8343   0.836011           0            1
97 2025-06-12 21:00:00  0.8335   0.834678           0            1
98 2025-06-12 21:15:00  0.8375   0.834234           0            1
99 2025-06-12 21:30:00  0.8427   0.835420           1            0


Buy signal detected for symbols: ['ONDOUSDT']


In [6]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()

2025-06-12 20:48:25,857 - INFO - Simple ATR Trading Bot initialized successfully
2025-06-12 20:48:25,858 - INFO - Initializing Binance Real exchange
2025-06-12 20:48:25,865 - INFO - 🚀 BUY SIGNAL: BTC/USDT
2025-06-12 20:48:25,865 - INFO - 💰 Investment Amount: 1000 USDT
2025-06-12 20:48:25,865 - INFO - 🌐 Trading on Real account
2025-06-12 20:48:29,082 - ERROR - ❌ Insufficient balance. Available: 0.00747323 USDT, Required: 1000 USDT


In [8]:
status = bot.get_position_status()
status

{}

In [6]:
oversold_15m = get_low_rsi_coins(rsi_threshold=30, interval="15m", lookback_hours=4)
oversold_15m

🚀 Fetching coins with RSI below 30
📊 Interval: 15m, Lookback: 4 hours
📈 Analyzing 403 USDT pairs...
✓ LINKUSDT: RSI 29.07, 4h Change: -1.50%
✓ ONEUSDT: RSI 23.58, 4h Change: -3.08%
✓ DUSKUSDT: RSI 28.86, 4h Change: -2.35%
✓ WINUSDT: RSI 24.74, 4h Change: -0.81%
✓ CHRUSDT: RSI 29.08, 4h Change: -1.83%
   Processed 100/403 coins... (Errors: 0)
✓ FLMUSDT: RSI 26.97, 4h Change: -4.37%
✓ 1INCHUSDT: RSI 29.58, 4h Change: -1.44%
✓ PHAUSDT: RSI 25.55, 4h Change: -2.07%
✓ C98USDT: RSI 27.77, 4h Change: -1.17%
✓ FLOWUSDT: RSI 27.40, 4h Change: -1.33%
✓ MOVRUSDT: RSI 29.02, 4h Change: -1.72%
✓ QIUSDT: RSI 26.16, 4h Change: -2.14%
   Processed 200/403 coins... (Errors: 0)
✓ FXSUSDT: RSI 22.33, 4h Change: -2.77%
✓ STEEMUSDT: RSI 26.40, 4h Change: -1.75%
✓ STGUSDT: RSI 27.20, 4h Change: -1.26%
✓ LUNCUSDT: RSI 27.53, 4h Change: -2.20%
✓ RPLUSDT: RSI 23.25, 4h Change: -5.49%
✓ LQTYUSDT: RSI 26.01, 4h Change: -2.74%
✓ RDNTUSDT: RSI 29.58, 4h Change: -2.61%
✓ MAVUSDT: RSI 29.09, 4h Change: -2.00%
✓ SEIU

,symbol,coin,current_price,rsi,interval,lookback_hours,change_4h,volume_4h,high_4h,low_4h,price_from_low,price_from_high
0,CYBERUSDT,CYBER,1.280000,21.146840,15m,4,-5.044510,5.783272e+05,1.356000,1.271000,0.708104,-5.604720
1,FXSUSDT,FXS,2.738000,22.328851,15m,4,-2.769886,1.406371e+05,2.830000,2.726000,0.440205,-3.250883
2,RPLUSDT,RPL,6.540000,23.249181,15m,4,-5.491329,1.495657e+05,6.910000,6.510000,0.460829,-5.354559
3,ONEUSDT,ONE,0.011320,23.584205,15m,4,-3.082192,3.122674e+07,0.011710,0.011320,0.000000,-3.330487
4,WINUSDT,WIN,0.000051,24.739629,15m,4,-0.809717,2.003190e+09,0.000052,0.000051,0.253313,-0.924321
5,PHAUSDT,PHA,0.118400,25.551881,15m,4,-2.067825,2.632697e+06,0.121800,0.118000,0.338983,-2.791461
6,LQTYUSDT,LQTY,1.063000,26.006789,15m,4,-2.744739,3.804330e+05,1.101000,1.054000,0.853890,-3.451408
7,QIUSDT,QI,0.007790,26.164826,15m,4,-2.135678,2.307085e+07,0.008040,0.007770,0.257400,-3.109453
8,STEEMUSDT,STEEM,0.134600,26.400621,15m,4,-1.751825,8.891674e+05,0.137200,0.134400,0.148810,-1.895044
9,FLMUSDT,FLM,0.035000,26.971380,15m,4,-4.371585,2.219439e+07,0.036700,0.034800,0.574713,-4.632153


In [ ]:
analyzer = ConsolidationAnalyzer()
non_consolidating_coins = analyzer.analyze_all_coins( 
        top_count=250  # Get top 100, display top 25
    )
con_df = analyzer.display_results(non_consolidating_coins, show_count=100)
con_df

Step 1: Fetching all trading pairs...
Found 403 active USDT trading pairs
Step 2: Fetching 24h ticker data...
Step 3: Analyzing 403 coins for consolidation...
Progress: 50/403 coins analyzed...
Progress: 100/403 coins analyzed...
Progress: 150/403 coins analyzed...
Progress: 200/403 coins analyzed...
Progress: 250/403 coins analyzed...
Progress: 300/403 coins analyzed...
Progress: 350/403 coins analyzed...
Progress: 400/403 coins analyzed...
Analysis complete! Processed 403 coins, 0 failed
Found 176 non-consolidating coins


,rank,symbol,price,24h_Change,24h_Range,volatility,ATR %,24h_Volume,vol_Surge
0,1,PEPEUSDT,$0.000012,-3.60%,10.16%,1.29%,1.93%,"$309,397,520",1.0x
1,2,KAIAUSDT,$0.167700,13.31%,19.56%,2.27%,3.56%,"$51,605,305",0.9x
2,3,HMSTRUSDT,$0.001108,-25.24%,39.17%,2.43%,4.01%,"$25,899,842",1.4x
3,4,TRXUSDT,$0.278000,-3.94%,5.72%,0.62%,0.72%,"$154,236,201",0.9x
4,5,RESOLVUSDT,$0.331300,9.92%,38.79%,2.89%,9.15%,"$54,111,655",0.4x
5,6,WIFUSDT,$0.942000,-6.92%,12.53%,1.44%,2.40%,"$76,014,709",1.0x
6,7,PENDLEUSDT,$3.908000,-11.92%,16.53%,1.26%,1.94%,"$27,919,868",1.4x
7,8,RVNUSDT,$0.021490,8.10%,22.43%,2.95%,4.55%,"$40,968,036",0.7x
8,9,ENAUSDT,$0.343900,-3.80%,10.73%,1.29%,2.15%,"$81,064,094",0.9x
9,10,ETHFIUSDT,$1.251000,-8.62%,12.71%,1.26%,2.31%,"$35,502,253",1.0x


In [7]:
calculator = BinanceVolatilityCalculator()
top_volatile = calculator.get_top_volatile_coins_4h(top_n=50)
top_volatile

Fetching last 4 hours of 15-minute data for volatility calculation...
Analysis time: 2025-06-12 11:20:27
Found 403 USDT pairs
Processed 20/403 pairs... (5.0%)
Processed 40/403 pairs... (9.9%)
Processed 60/403 pairs... (14.9%)
Processed 80/403 pairs... (19.9%)
Processed 100/403 pairs... (24.8%)
Processed 120/403 pairs... (29.8%)
Processed 140/403 pairs... (34.7%)
Processed 160/403 pairs... (39.7%)
Processed 180/403 pairs... (44.7%)
Processed 200/403 pairs... (49.6%)
Processed 220/403 pairs... (54.6%)
Processed 240/403 pairs... (59.6%)
Processed 260/403 pairs... (64.5%)
Processed 280/403 pairs... (69.5%)
Processed 300/403 pairs... (74.4%)
Processed 320/403 pairs... (79.4%)
Processed 340/403 pairs... (84.4%)
Processed 360/403 pairs... (89.3%)
Processed 380/403 pairs... (94.3%)
Processed 400/403 pairs... (99.3%)

Analysis complete! Found 400 qualifying coins.


,symbol,coin,current_price,start_price_4h,price_change_4h,volatility_score,std_volatility,price_range_volatility,max_single_move,max_drawdown_4h,volume_4h,volume_volatility,data_points,rank
402,RESOLVUSDT,RESOLV,3.438000e-01,3.851000e-01,10.724487,3.165264,5.361615,3.774451,6.030429,16.277298,1.471123e+08,0.012627,48,1
69,ARDRUSDT,ARDR,9.550000e-02,8.566000e-02,11.487275,3.046962,5.281250,1.288103,9.230406,3.114538,3.363698e+07,0.030856,48,2
47,RVNUSDT,RVN,2.130000e-02,2.210000e-02,3.619910,2.789526,5.408317,2.236924,6.422018,11.824755,9.779415e+08,0.016300,48,3
333,HMSTRUSDT,HMSTR,1.065000e-03,1.283000e-03,16.991426,2.408583,4.416645,1.926811,5.747126,18.139892,1.261090e+10,0.015592,48,4
361,ANIMEUSDT,ANIME,3.143000e-02,3.013000e-02,4.314637,1.858972,4.024325,1.880184,3.371522,4.623179,5.320202e+08,0.012996,48,5
56,FTTUSDT,FTT,9.703000e-01,9.592000e-01,1.157214,1.668508,3.157242,1.104118,4.251889,4.251889,1.826376e+06,0.015228,48,6
385,VIRTUALUSDT,VIRTUAL,2.071400e+00,2.248400e+00,7.872265,1.573198,2.938396,1.306222,3.640929,10.383317,1.566053e+07,0.008778,48,7
156,MASKUSDT,MASK,1.578000e+00,1.565000e+00,0.830671,1.536301,3.067381,1.438507,3.158560,7.880911,1.391644e+07,0.009065,48,8
107,FLMUSDT,FLM,3.690000e-02,3.750000e-02,1.600000,1.490171,2.659364,1.043112,3.835616,4.800000,4.024932e+07,0.008700,48,9
159,ATAUSDT,ATA,5.040000e-02,5.080000e-02,0.787402,1.464923,2.416251,0.903150,4.106776,5.458090,1.514126e+07,0.027749,48,10
